In [ ]:
# Uncomment the below line if using Kaggle P100
#!pip install --force-reinstall torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import os
import math
import random
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report, roc_curve
from tqdm.auto import tqdm

# Config - Paths
DATA_DIR  = Path("/kaggle/input/competitions/siim-isic-melanoma-classification")
CSV_PATH  = DATA_DIR / "train.csv"
TEST_CSV_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

IMAGE_DIR = DATA_DIR / "jpeg/train"
TEST_IMAGE_DIR = DATA_DIR / "jpeg/test"

OUTPUT_DIR = Path("/kaggle/working/outputs_siim_isic_b7")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config - Hyperparameters
SEED = 42
VALID_FRACTION = 0.10

MODEL_NAME = "tf_efficientnet_b7_ns"
IMAGE_SIZE = 512
DROPOUT = 0.30
META_HIDDEN = 128

NUM_EPOCHS = 8
BATCH_SIZE = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4
MIN_LR = 1e-6
GRAD_CLIP = 1.0

POSITIVE_OVERSAMPLE_MULTIPLIER = 4.0

USE_AMP = True
CHANNELS_LAST = True

NUM_WORKERS = min(4, os.cpu_count() or 2)
EVAL_NUM_WORKERS = 2
PIN_MEMORY = True

TTA_MODES = ("none", "hflip", "vflip", "hvflip")

BEST_MODEL_PATH = OUTPUT_DIR / "best_model_b7_joint_meta.pth"
HISTORY_PATH = OUTPUT_DIR / "history_b7_joint_meta.csv"
VALID_PRED_PATH = OUTPUT_DIR / "valid_predictions_b7_joint_meta.csv"
TEST_PRED_PATH = OUTPUT_DIR / "test_predictions_b7_joint_meta.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission_b7_joint_meta.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CSV exists:",          CSV_PATH.exists())
print("Image folder exists:", IMAGE_DIR.exists())
print("CUDA available:",      torch.cuda.is_available())

In [ ]:
# Set all seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

df = pd.read_csv(CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print(df.head())
print(df.columns)
print("\nClass distribution:")
print(df["target"].value_counts())
print("\nTest rows:")
print(len(test_df))

In [ ]:
META_NUM_COLS = ["age_approx", "n_images", "image_size"]
META_CAT_COLS = ["sex", "anatom_site_general_challenge"]


def add_meta_columns(data, image_dir):
    data = data.copy()

    data["image_path"] = data["image_name"].apply(lambda x: image_dir / f"{x}.jpg")

    data["sex"] = data["sex"].fillna("unknown").astype(str).str.lower()
    data["anatom_site_general_challenge"] = (
        data["anatom_site_general_challenge"]
        .fillna("unknown")
        .astype(str)
        .str.lower()
    )

    data["age_approx"] = pd.to_numeric(data["age_approx"], errors="coerce")
    data["age_missing"] = data["age_approx"].isna().astype(np.float32)

    data["n_images"] = data.groupby("patient_id")["image_name"].transform("count")
    data["n_images"] = data["n_images"].astype(np.float32)

    data["image_size"] = data["image_path"].apply(
        lambda p: os.path.getsize(p) if Path(p).exists() else 0
    ).astype(np.float32)

    return data


def fit_meta_features(train_df, val_df, test_df):
    train_df = train_df.copy()
    val_df = val_df.copy()
    test_df = test_df.copy()

    age_median = train_df["age_approx"].median()

    for data in (train_df, val_df, test_df):
        data["age_approx"] = data["age_approx"].fillna(age_median)

    scaler = StandardScaler()
    train_num = scaler.fit_transform(train_df[META_NUM_COLS])
    val_num = scaler.transform(val_df[META_NUM_COLS])
    test_num = scaler.transform(test_df[META_NUM_COLS])

    train_cat = pd.get_dummies(train_df[META_CAT_COLS]).astype(np.float32)
    val_cat = pd.get_dummies(val_df[META_CAT_COLS]).reindex(
        columns=train_cat.columns, fill_value=0
    ).astype(np.float32)
    test_cat = pd.get_dummies(test_df[META_CAT_COLS]).reindex(
        columns=train_cat.columns, fill_value=0
    ).astype(np.float32)

    meta_features = META_NUM_COLS + list(train_cat.columns) + ["age_missing"]

    train_meta = np.c_[train_num, train_cat.values, train_df["age_missing"].values]
    val_meta = np.c_[val_num, val_cat.values, val_df["age_missing"].values]
    test_meta = np.c_[test_num, test_cat.values, test_df["age_missing"].values]

    for data, meta in [(train_df, train_meta), (val_df, val_meta), (test_df, test_meta)]:
        data[meta_features] = meta.astype(np.float32)

    return train_df, val_df, test_df, meta_features


meta_df = add_meta_columns(df, IMAGE_DIR)
test_df = add_meta_columns(test_df, TEST_IMAGE_DIR)

patient_targets = meta_df.groupby("patient_id")["target"].max().reset_index()

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=VALID_FRACTION,
    random_state=SEED,
)

train_idx, val_idx = next(
    splitter.split(patient_targets["patient_id"], patient_targets["target"])
)

train_patients = set(patient_targets.iloc[train_idx]["patient_id"])
val_patients = set(patient_targets.iloc[val_idx]["patient_id"])

train_df = meta_df[meta_df["patient_id"].isin(train_patients)].reset_index(drop=True)
val_df = meta_df[meta_df["patient_id"].isin(val_patients)].reset_index(drop=True)

train_df, val_df, test_df, META_FEATURES = fit_meta_features(train_df, val_df, test_df)

print("Train rows:", len(train_df), "| patients:", train_df["patient_id"].nunique())
print("Val rows:", len(val_df), "| patients:", val_df["patient_id"].nunique())
print("Test rows:", len(test_df))
print("\nTrain targets:\n", train_df["target"].value_counts())
print("\nVal targets:\n", val_df["target"].value_counts())
print("\nMetadata features:", len(META_FEATURES))

In [ ]:
class RandomHair(A.ImageOnlyTransform):
    def __init__(self, p=0.25):
        super().__init__(p=p)

    def apply(self, image, **params):
        image = image.copy()
        h, w = image.shape[:2]

        for _ in range(random.randint(1, 6)):
            x1, y1 = random.randint(0, w - 1), random.randint(0, h - 1)
            x2 = np.clip(x1 + random.randint(-w // 3, w // 3), 0, w - 1)
            y2 = np.clip(y1 + random.randint(-h // 3, h // 3), 0, h - 1)
            cv2.line(image, (x1, y1), (x2, y2), (0, 0, 0), random.randint(1, 2))

        return image


def coarse_dropout():
    return A.CoarseDropout(
        num_holes_range=(1, 6),
        hole_height_range=(0.03, 0.10),
        hole_width_range=(0.03, 0.10),
        fill=0,
        p=0.35,
    )


train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),

    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Transpose(p=0.2),
    A.ShiftScaleRotate(
        shift_limit=0.08,
        scale_limit=0.10,
        rotate_limit=45,
        border_mode=cv2.BORDER_REFLECT_101,
        p=0.5,
    ),

    A.RandomBrightnessContrast(0.12, 0.12, p=0.35),
    A.HueSaturationValue(8, 10, 8, p=0.25),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.15),

    A.OneOf([
        A.MotionBlur(blur_limit=5),
        A.MedianBlur(blur_limit=5),
        A.GaussianBlur(blur_limit=(3, 5)),
    ], p=0.20),

    A.OneOf([
        A.GaussNoise(),
        A.ISONoise(),
    ], p=0.15),

    RandomHair(p=0.25),
    coarse_dropout(),

    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])


val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    ),
    ToTensorV2(),
])

In [ ]:
class MelanomaDataset(Dataset):
    def __init__(self, dataframe, image_dir, feature_cols, transform=None, is_test=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.feature_cols = feature_cols
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = row.get("image_path", self.image_dir / f"{row['image_name']}.jpg")
        image = Image.open(image_path).convert("RGB")
        image = np.array(image)

        if self.transform is not None:
            image = self.transform(image=image)["image"]

        meta = torch.tensor(
            row[self.feature_cols].values.astype(float),
            dtype=torch.float32,
        )

        if self.is_test:
            return image, meta, row["image_name"]

        label = torch.tensor(row["target"], dtype=torch.float32)
        return image, meta, label

In [ ]:
train_set = MelanomaDataset(train_df, IMAGE_DIR, META_FEATURES, transform=train_transform)
val_set = MelanomaDataset(val_df, IMAGE_DIR, META_FEATURES, transform=val_transform)
test_set = MelanomaDataset(test_df, TEST_IMAGE_DIR, META_FEATURES, transform=val_transform, is_test=True)


sample_weights = np.where(
    train_df["target"].values == 1,
    POSITIVE_OVERSAMPLE_MULTIPLIER,
    1.0,
).astype(np.float32)

train_sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)


train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY and device.type == "cuda",
    persistent_workers=NUM_WORKERS > 0,
)

val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=EVAL_NUM_WORKERS,
    pin_memory=PIN_MEMORY and device.type == "cuda",
    persistent_workers=EVAL_NUM_WORKERS > 0,
)

test_loader = DataLoader(
    test_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=EVAL_NUM_WORKERS,
    pin_memory=PIN_MEMORY and device.type == "cuda",
    persistent_workers=EVAL_NUM_WORKERS > 0,
)

print("Train samples:", len(train_set))
print("Val samples:", len(val_set))
print("Test samples:", len(test_set))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
images, metas, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Meta  batch shape:", metas.shape)
print("Label batch shape:", labels.shape)
print("First 10 labels:",   labels[:10])

In [ ]:
class MelanomaModel(nn.Module):
    def __init__(self, model_name=MODEL_NAME, meta_dim=None, n_classes=1):
        super().__init__()

        if meta_dim is None:
            meta_dim = len(META_FEATURES)

        self.backbone = timm.create_model(
            model_name,
            pretrained=True,
            num_classes=0,
            global_pool="avg",
        )

        img_features = self.backbone.num_features

        self.meta_branch = nn.Sequential(
            nn.Linear(meta_dim, META_HIDDEN),
            nn.BatchNorm1d(META_HIDDEN),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(META_HIDDEN, META_HIDDEN),
            nn.BatchNorm1d(META_HIDDEN),
            nn.ReLU(inplace=True),
        )

        self.head = nn.Sequential(
            nn.Linear(img_features + META_HIDDEN, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(256, n_classes),
        )

    def forward(self, image, meta):
        img_feat = self.backbone(image)
        meta_feat = self.meta_branch(meta)
        fused = torch.cat([img_feat, meta_feat], dim=1)
        return self.head(fused).squeeze(1)

In [ ]:
scaler = GradScaler(enabled=USE_AMP and device.type == "cuda")
print("Using device:", device)

In [ ]:
model = MelanomaModel(
    model_name=MODEL_NAME,
    meta_dim=len(META_FEATURES),
    n_classes=1,
).to(device)

if CHANNELS_LAST and device.type == "cuda":
    model = model.to(memory_format=torch.channels_last)


n_pos = train_df["target"].sum()
n_neg = len(train_df) - n_pos
pos_weight = torch.tensor([n_neg / max(1, n_pos)], dtype=torch.float32, device=device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler(enabled=USE_AMP and device.type == "cuda")


def make_scheduler(optimizer):
    total_steps = NUM_EPOCHS * len(train_loader)
    warmup_steps = max(len(train_loader), int(0.1 * total_steps))

    def schedule(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        return (MIN_LR / LR) + (1 - MIN_LR / LR) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, schedule)


def send_batch(images, metas, labels=None):
    images = images.to(device, non_blocking=True)
    metas = metas.to(device, non_blocking=True)

    if CHANNELS_LAST and device.type == "cuda":
        images = images.contiguous(memory_format=torch.channels_last)

    if labels is None:
        return images, metas

    return images, metas, labels.to(device, non_blocking=True)


@torch.no_grad()
def run_validation(model):
    model.eval()

    losses, probs, targets = [], [], []

    for images, metas, labels in tqdm(val_loader, desc="Valid", leave=False):
        images, metas, labels_gpu = send_batch(images, metas, labels)

        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(images, metas)
            loss = criterion(logits, labels_gpu)

        losses.append(loss.item())
        probs.extend(torch.sigmoid(logits).cpu().numpy())
        targets.extend(labels.numpy())

    probs = np.array(probs)
    targets = np.array(targets)

    auc = roc_auc_score(targets, probs)
    acc = accuracy_score(targets, (probs >= 0.5).astype(int))

    return np.mean(losses), auc, acc


scheduler = make_scheduler(optimizer)
best_auc = 0.0
history = []


for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_losses = []

    for images, metas, labels in tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False):
        images, metas, labels = send_batch(images, metas, labels)

        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=USE_AMP and device.type == "cuda"):
            logits = model(images, metas)
            loss = criterion(logits, labels)

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss at epoch {epoch}: {loss.item()}")

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        train_losses.append(loss.item())

    val_loss, val_auc, val_acc = run_validation(model)

    row = {
        "epoch": epoch,
        "train_loss": np.mean(train_losses),
        "val_loss": val_loss,
        "val_auc": val_auc,
        "val_acc": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={row['train_loss']:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_auc={val_auc:.4f} | "
        f"val_acc={val_acc:.4f}"
    )

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  saved best model: {best_auc:.4f}")


history_df = pd.DataFrame(history)
history_df.to_csv(HISTORY_PATH, index=False)

print(f"Training complete. Best AUC: {best_auc:.4f}")

In [ ]:
def tta_transform(images, mode):
    if mode == "none":
        return images
    if mode == "hflip":
        return images.flip(3)
    if mode == "vflip":
        return images.flip(2)
    if mode == "hvflip":
        return images.flip(2).flip(3)

    raise ValueError(f"Unknown TTA mode: {mode}")


def predict_val_loader(model, loader):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, metas, labels in tqdm(loader, desc="Validation TTA", leave=False):
            images = images.to(device, non_blocking=True)
            metas = metas.to(device, non_blocking=True)

            if CHANNELS_LAST and device.type == "cuda":
                images = images.contiguous(memory_format=torch.channels_last)

            tta_probs = torch.zeros(images.size(0), device=device)

            for mode in TTA_MODES:
                aug_images = tta_transform(images, mode)
                logits = model(aug_images, metas)
                tta_probs += torch.sigmoid(logits)

            tta_probs /= len(TTA_MODES)

            all_probs.extend(tta_probs.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_probs), np.array(all_labels)


model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

val_probs, true_labels = predict_val_loader(model, val_loader)

val_auc = roc_auc_score(true_labels, val_probs)
val_preds = (val_probs >= 0.5).astype(int)

print(f"Validation AUC with TTA x{len(TTA_MODES)}: {val_auc:.4f}")
print(classification_report(true_labels, val_preds, target_names=["Benign", "Melanoma"]))

valid_pred_df = val_df[["image_name", "target", "patient_id"]].copy()
valid_pred_df["pred"] = val_probs
valid_pred_df.to_csv(VALID_PRED_PATH, index=False)

print("Saved validation predictions:", VALID_PRED_PATH)

In [ ]:
cm = confusion_matrix(true_labels, val_preds)

print("Classification Report:")
print(classification_report(true_labels, val_preds, target_names=["Benign", "Melanoma"]))
print(f"\nB7 joint model AUC: {val_auc:.4f}")

plt.figure(figsize=(5, 4))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix - B7")
plt.xticks([0, 1], ["Benign", "Melanoma"])
plt.yticks([0, 1], ["Benign", "Melanoma"])
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="black", fontsize=12)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_b7_joint_meta.png", dpi=150)
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(true_labels, val_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"B7 joint model (AUC = {val_auc:.4f})", lw=2)
plt.plot([0, 1], [0, 1], "k--", lw=1, label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - B7 Joint Image+Metadata Model")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve_b7_joint_meta.png", dpi=150)
plt.show()

print("ROC curve saved.")

In [ ]:
# Add test prediction + submission (in case we want to submit to Kaggle)
def predict_test_loader(model, loader):
    model.eval()

    all_probs = []
    all_names = []

    with torch.no_grad():
        for images, metas, names in tqdm(loader, desc="Test TTA", leave=False):
            images = images.to(device, non_blocking=True)
            metas = metas.to(device, non_blocking=True)

            if CHANNELS_LAST and device.type == "cuda":
                images = images.contiguous(memory_format=torch.channels_last)

            tta_probs = torch.zeros(images.size(0), device=device)

            for mode in TTA_MODES:
                aug_images = tta_transform(images, mode)
                logits = model(aug_images, metas)
                tta_probs += torch.sigmoid(logits)

            tta_probs /= len(TTA_MODES)

            all_probs.extend(tta_probs.cpu().numpy())
            all_names.extend(names)

    return np.array(all_names), np.array(all_probs)


test_names, test_probs = predict_test_loader(model, test_loader)

test_pred_df = pd.DataFrame({
    "image_name": test_names,
    "target": test_probs,
})

test_pred_df.to_csv(TEST_PRED_PATH, index=False)

submission = sample_sub.drop(columns=["target"], errors="ignore").merge(
    test_pred_df,
    on="image_name",
    how="left",
)

submission["target"] = submission["target"].fillna(0.0)
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved test predictions:", TEST_PRED_PATH)
print("Saved submission:", SUBMISSION_PATH)
print(submission.head())

In [ ]:
def get_gradcam(model, image_tensor, meta_tensor):
    model.eval()
    gradients, activations = [], []

    target_layer = model.backbone.blocks[-1]

    def forward_hook(module, input, output):
        activations.append(output)

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0])

    fh = target_layer.register_forward_hook(forward_hook)
    bh = target_layer.register_full_backward_hook(backward_hook)

    image_tensor = image_tensor.unsqueeze(0).to(device)
    meta_tensor = meta_tensor.unsqueeze(0).to(device)

    if CHANNELS_LAST and device.type == "cuda":
        image_tensor = image_tensor.contiguous(memory_format=torch.channels_last)

    logit = model(image_tensor, meta_tensor)

    model.zero_grad()
    logit[0].backward()

    fh.remove()
    bh.remove()

    grads = gradients[0]
    acts = activations[0]
    weights = grads.mean(dim=[2, 3], keepdim=True)

    cam = (weights * acts).sum(dim=1).squeeze()
    cam = F.relu(cam)
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    return cam.detach().cpu().numpy()


def show_gradcam(model, val_df, val_transform, n_samples=6):
    melanoma_rows = val_df[val_df["target"] == 1].sample(n_samples // 2, random_state=0)
    benign_rows = val_df[val_df["target"] == 0].sample(n_samples // 2, random_state=0)
    samples = pd.concat([melanoma_rows, benign_rows]).sample(frac=1, random_state=0)

    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples * 3, 6))

    for col, (_, row) in enumerate(samples.iterrows()):
        img_path = row["image_path"]

        orig_img = Image.open(img_path).convert("RGB").resize((224, 224))

        image_np = np.array(Image.open(img_path).convert("RGB"))
        image_tensor = val_transform(image=image_np)["image"]

        meta_tensor = torch.tensor(
            row[META_FEATURES].values.astype(float),
            dtype=torch.float32,
        )

        cam = get_gradcam(model, image_tensor, meta_tensor)

        cam_resized = cv2.resize(cam, (224, 224))
        heatmap = plt.cm.jet(cam_resized)[:, :, :3]
        overlay = np.array(orig_img) / 255.0 * 0.5 + heatmap * 0.5

        true_label = "Melanoma" if row["target"] == 1 else "Benign"

        axes[0, col].imshow(orig_img)
        axes[0, col].set_title(f"True: {true_label}", fontsize=9)
        axes[0, col].axis("off")

        axes[1, col].imshow(overlay)
        axes[1, col].set_title("Grad-CAM", fontsize=9)
        axes[1, col].axis("off")

    plt.suptitle("Grad-CAM: B7 joint model attention regions", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "gradcam_b7_joint_meta.png", dpi=150)
    plt.show()


show_gradcam(model, val_df, val_transform, n_samples=6)